# 🎭 Face Recognition + Sentiment Detection
### Detects **Menna** and labels everyone else as **Unknown** — with live emotion analysis

**Stack:** YOLOv8 (face detection) + DeepFace (identity + emotion)  
**Steps:**
1. Install dependencies
2. Enroll Menna's face (take 5 reference photos)
3. Run live webcam detection

## Cell 1 — Install dependencies

In [ ]:
# Install all required packages
!pip install ultralytics deepface tf-keras opencv-python-headless ipywidgets -q
!pip install retina-face -q

import sys, subprocess
print('✅ Installation complete')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.6 MB/s eta 0:00:00
✅ Installation complete


## Cell 2 — Imports & setup

In [2]:
import cv2
import numpy as np
import os
import time
import base64
from pathlib import Path
from IPython.display import display, Javascript, Image, clear_output
import ipywidgets as widgets
from google.colab.output import eval_js
from ultralytics import YOLO
from deepface import DeepFace
import warnings
warnings.filterwarnings('ignore')

# Create folder to store Menna's reference images
MENNA_DB = '/content/menna_db/Menna'
os.makedirs(MENNA_DB, exist_ok=True)

# Emotion → emoji map
EMOTION_EMOJI = {
    'happy':   '😊 Happy',
    'sad':     '😢 Sad',
    'angry':   '😠 Angry',
    'surprise':'😲 Surprise',
    'fear':    '😨 Fear',
    'disgust': '🤢 Disgust',
    'neutral': '😐 Neutral'
}

print('✅ Imports done')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
26-04-28 08:15:58 - Directory /root/.deepface has been created
26-04-28 08:15:58 - Directory /root/.deepface/weights has been created


NameError: name 'menna_DB' is not defined

## Cell 3 — Download YOLOv8 face detection model
We use a YOLOv8n model fine-tuned on faces (faster & more accurate than generic detection)

In [ ]:
from retinaface import RetinaFace
import torch

# Pre-warm the model
print("Initializing RetinaFace detector...")
# RetinaFace handles its own weight downloads and caching
test_img = np.zeros((100, 100, 3), dtype=np.uint8)
_ = RetinaFace.detect_faces(test_img)

print('\n✅ Face detection model (RetinaFace) ready')

## Cell 4 — Enroll Menna's face
**Run this cell to capture 5 reference photos of Menna.**  
Face the camera directly, with good lighting. These photos will be used for identity matching.

In [ ]:
from google.colab import files
import shutil

print('Upload 3–10 clear photos of Menna (face clearly visible, good lighting)')
uploaded = files.upload()

for i, (fname, data) in enumerate(uploaded.items()):
    dest = f'{MENNA_DB}/menna_upload_{i:02d}.jpg'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved: {dest}')

total = len(list(Path(MENNA_DB).glob('*.jpg')))
print(f'\n✅ menna DB now has {total} reference images')

# Clear DeepFace's representation cache so it re-indexes
cache_file = '/content/WhatsApp Image 2026-04-28 at 11.12.25 AM.jpeg'
if os.path.exists(cache_file):
    os.remove(cache_file)
    print('🔄 Cache cleared — DB will re-index on next run')

## Cell 5 — Helper functions

In [ ]:
from retinaface import RetinaFace

def b64_to_cv2(b64_str):
    """Convert base64 image string to OpenCV array."""
    img_data = base64.b64decode(b64_str.split(',')[1])
    arr = np.frombuffer(img_data, np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)

def cv2_to_b64(frame):
    """Convert OpenCV frame to base64 string for display."""
    _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return base64.b64encode(buf).decode('utf-8')

def detect_faces_yolo(frame, model=None, conf=0.4):
    """Run RetinaFace detection. Returns list of (x1,y1,x2,y2) boxes."""
    # Note: Function name kept for compatibility with subsequent cells
    try:
        faces = RetinaFace.detect_faces(frame)
        boxes = []
        if isinstance(faces, dict):
            for key in faces:
                face = faces[key]
                area = face['facial_area'] # [x1, y1, x2, y2]
                boxes.append((int(area[0]), int(area[1]), int(area[2]), int(area[3])))
        return boxes
    except:
        return []

def recognize_identity(face_crop):
    """Compare face crop against Menna's DB. Returns 'Menna' or 'Unknown'."""
    try:
        result = DeepFace.find(
            img_path=face_crop,
            db_path='/content/menna_db',
            model_name='Facenet512',
            enforce_detection=False,
            silent=True
        )
        if result and len(result[0]) > 0:
            dist_col = [c for c in result[0].columns if 'distance' in c.lower()]
            if dist_col and result[0][dist_col[0]].iloc[0] < 0.45:
                return 'Menna'
        return 'Unknown'
    except Exception:
        return 'Unknown'

def detect_emotion(face_crop):
    """Detect dominant emotion from face crop."""
    try:
        analysis = DeepFace.analyze(
            img_path=face_crop,
            actions=['emotion'],
            enforce_detection=False,
            silent=True
        )
        emotion = analysis[0]['dominant_emotion'].lower()
        return EMOTION_EMOJI.get(emotion, emotion.capitalize())
    except Exception:
        return ''

def draw_annotation(frame, box, name, emotion):
    """Draw bounding box + label on frame."""
    x1, y1, x2, y2 = box
    color = (0, 200, 80) if name == 'Menna' else (50, 50, 220)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    label = name
    (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    cv2.rectangle(frame, (x1, y1 - lh - 12), (x1 + lw + 8, y1), color, -1)
    cv2.putText(frame, label, (x1 + 4, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    if emotion:
        emotion_text = emotion.split(' ', 1)[-1] if ' ' in emotion else emotion
        cv2.putText(frame, emotion_text, (x1, y2 + 22), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return frame

print('✅ Helper functions updated with RetinaFace')

## Cell 6 — Live webcam detection
**Run this cell to start real-time face recognition + sentiment detection.**  
- 🟢 **Menna** → green box
- 🔴 **Unknown** → red/blue box  
- Emotion is displayed below each box

Press the **Stop** button to end the session.

## Cell 9 — Live Webcam (Library Approach)
This implementation uses a standard JavaScript-to-Python bridge to process frames continuously.

In [ ]:
import base64
import html
import io
import time
import cv2
import numpy as np
from IPython.display import display, Javascript, clear_output
import ipywidgets as widgets
from google.colab.output import eval_js

def video_stream():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;

    var pendingResolve = null;
    var shutdown = false;

    async function cleanup() {
      if (window._currentStream) {
        window._currentStream.getTracks().forEach(t => t.stop());
      }
    }

    function removeDom() {
       if (stream) stream.getTracks().forEach(t => t.stop());
       if (video) video.remove();
       if (div) div.remove();
       video = null;
       div = null;
       stream = null;
    }

    function onAnimationFrame() {
      if (!shutdown) {
        window.requestAnimationFrame(onAnimationFrame);
      }
      if (pendingResolve) {
        var result = ""
        if (!shutdown && video && video.readyState === 4) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }

    window.createDom = async function() {
      if (div !== null) return;
      await cleanup();

      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.width = '640px';
      div.style.margin = '10px auto';

      video = document.createElement('video');
      video.style.display = 'block';
      video.width = 640;
      video.height = 480;
      div.appendChild(video);

      const stopBtn = document.createElement('button');
      stopBtn.innerText = 'Stop Webcam';
      stopBtn.style.width = '100%';
      stopBtn.onclick = () => { shutdown = true; };
      div.appendChild(stopBtn);

      document.body.appendChild(div);

      try {
        stream = await navigator.mediaDevices.getUserMedia({video: true});
        window._currentStream = stream;
        video.srcObject = stream;
        await video.play();

        captureCanvas = document.createElement('canvas');
        captureCanvas.width = 640;
        captureCanvas.height = 480;

        window.requestAnimationFrame(onAnimationFrame);
      } catch (e) {
        alert('Camera error: ' + e.message);
      }
    };

    window.getFrame = async function() {
      if (!window.createDom) return "";
      await window.createDom();
      if (shutdown) {
        removeDom();
        shutdown = false;
        return "";
      }
      return new Promise(resolve => { pendingResolve = resolve; });
    };
    ''')
  display(js)

def video_frame():
  return eval_js('window.getFrame()')

# Initialize UI
clear_output()
video_stream()
time.sleep(2.0)

# Use ipywidgets for a smoother 'live' feeling display
image_widget = widgets.Image(format='jpeg', width=640, height=480)
display(image_widget)

try:
    while True:
        frame_js = video_frame()
        if not frame_js: break

        img = b64_to_cv2(frame_js)
        if img is None: continue

        # Process frame
        boxes = detect_faces_yolo(img)
        for box in boxes:
            x1, y1, x2, y2 = box
            face_crop = img[y1:y2, x1:x2]
            if face_crop.size > 0:
                name = recognize_identity(face_crop)
                emotion = detect_emotion(face_crop)
                img = draw_annotation(img, box, name, emotion)

        # Update persistent widget instead of clearing entire output
        _, encoded_img = cv2.imencode('.jpg', img)
        image_widget.value = bytes(encoded_img)

except KeyboardInterrupt:
    print("Stream stopped manually.")

## Selective Emotion Detection
This version only runs emotion analysis for **Menna** and ignores emotion for **Unknown** faces.

In [ ]:
# Initialize UI
clear_output()
video_stream()
time.sleep(2.0)

image_widget = widgets.Image(format='jpeg', width=640, height=480)
display(image_widget)

try:
    while True:
        frame_js = video_frame()
        if not frame_js: break

        img = b64_to_cv2(frame_js)
        if img is None: continue

        boxes = detect_faces_yolo(img)
        for box in boxes:
            x1, y1, x2, y2 = box
            face_crop = img[y1:y2, x1:x2]

            if face_crop.size > 0:
                name = recognize_identity(face_crop)

                # Only detect emotion if identity is Menna
                emotion = ""
                if name == "Menna":
                    emotion = detect_emotion(face_crop)

                img = draw_annotation(img, box, name, emotion)

        _, encoded_img = cv2.imencode('.jpg', img)
        image_widget.value = bytes(encoded_img)

except KeyboardInterrupt:
    print("Stream stopped manually.")

## Cell 7 — (Optional) Re-enroll Menna
 from uploaded images
If the live capture didn't work well, upload clear photos of yourself here.

In [ ]:
from google.colab import files
import shutil

print('Upload 3–10 clear photos of menna (face clearly visible, good lighting)')
uploaded = files.upload()

for i, (fname, data) in enumerate(uploaded.items()):
    dest = f'{menna_DB}/menna_upload_{i:02d}.jpg'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved: {dest}')

total = len(list(Path(menna_DB).glob('*.jpg')))
print(f'\n✅ menna DB now has {total} reference images')

# Clear DeepFace's representation cache so it re-indexes
cache_file = '/content/menna_db/representations_facenet512.pkl'
if os.path.exists(cache_file):
    os.remove(cache_file)
    print('🔄 Cache cleared — DB will re-index on next run')

## Cell 8 — (Optional) Test on a single image
Upload any photo to test identity + emotion detection before running live.

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt
import cv2

print('Upload a test image:')
uploaded = files.upload()

for fname, data in uploaded.items():
    test_path = f'/content/{fname}'
    with open(test_path, 'wb') as f:
        f.write(data)

    frame = cv2.imread(test_path)
    # Removed 'face_model' argument as detect_faces_yolo is self-contained
    boxes = detect_faces_yolo(frame)
    print(f'Detected {len(boxes)} face(s)')

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        crop = frame[y1:y2, x1:x2]
        name    = recognize_identity(crop)
        emotion = detect_emotion(crop)
        print(f'  Face {i+1}: {name} | {emotion}')
        frame = draw_annotation(frame, box, name, emotion)

    # Show result
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Recognition result')
    plt.tight_layout()
    plt.show()

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt
import cv2

print('Upload a test image:')
uploaded = files.upload()

for fname, data in uploaded.items():
    test_path = f'/content/{fname}'
    with open(test_path, 'wb') as f:
        f.write(data)

    frame = cv2.imread(test_path)
    # Removed 'face_model' argument as detect_faces_yolo is self-contained
    boxes = detect_faces_yolo(frame)
    print(f'Detected {len(boxes)} face(s)')

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        crop = frame[y1:y2, x1:x2]
        name    = recognize_identity(crop)
        emotion = detect_emotion(crop)
        print(f'  Face {i+1}: {name} | {emotion}')
        frame = draw_annotation(frame, box, name, emotion)

    # Show result
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Recognition result')
    plt.tight_layout()
    plt.show()